In [ ]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D3 — BLS Occupational Employment and Wages
# ============================================================

!pip install pymupdf -q

from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D3"

DOCUMENT_NAME = (
    "Occupational Employment and Wages — May 2024"
)

REFERENCE_PERIOD = "May 2024"

SOURCE_PAGE_START = 1
SOURCE_PAGE_END = 5

EXPECTED_REFERENCE_RECORD_COUNT = 70

REFERENCE_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Value",
    "Unit",
    "Reference Period",
    "Source Location"
]

ALLOWED_UNITS = {
    "workers",
    "million workers",
    "percent",
    "USD"
}

OUTPUT_DIR = Path(
    "outputs_D3_stage1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Document:", DOCUMENT_ID)
print("Reference period:", REFERENCE_PERIOD)
print(
    "Source scope:",
    f"Pages {SOURCE_PAGE_START}–{SOURCE_PAGE_END}"
)
print(
    "Expected reference records:",
    EXPECTED_REFERENCE_RECORD_COUNT
)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. Upload source document
# ============================================================

print(
    "Upload the original D3 PDF."
)

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError(
        "Upload exactly one PDF file."
    )

SOURCE_PATH = pdf_files[0]

print(
    "Loaded:",
    SOURCE_PATH.name
)

In [ ]:
# ============================================================
# 3. Load document and extract page text
# ============================================================

doc = fitz.open(
    SOURCE_PATH
)

page_texts = []

for page_index, page in enumerate(doc):

    page_texts.append({
        "Page Number":
            page_index + 1,

        "Text":
            page.get_text(
                "text"
            )
    })


full_text = "\n".join(
    page_record["Text"]
    for page_record in page_texts
)


if not full_text.strip():

    raise ValueError(
        "No extractable text was found in the PDF."
    )


print(
    "Pages:",
    len(doc)
)

print(
    "Extracted characters:",
    len(full_text)
)

In [ ]:
# ============================================================
# 4. Source metadata
# ============================================================

def sha256_file(path):
    """
    Return the SHA-256 hash of a file.
    """

    hash_object = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):

            hash_object.update(
                chunk
            )

    return hash_object.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


DOCUMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "file_name":
        SOURCE_PATH.name,

    "file_format":
        "PDF",

    "file_sha256":
        SOURCE_SHA256,

    "number_of_pages":
        len(doc),

    "text_extractable":
        bool(
            full_text.strip()
        ),

    "ocr_required":
        False,

    "total_text_characters":
        len(full_text),

    "total_text_words":
        len(
            full_text.split()
        ),

    "reference_source_pages":
        list(
            range(
                SOURCE_PAGE_START,
                SOURCE_PAGE_END + 1
            )
        ),

    "reference_period":
        REFERENCE_PERIOD,
}

print(
    json.dumps(
        DOCUMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 5. Page-level document profile
# ============================================================

page_characterisation = []

for page_record in page_texts:

    page_number = page_record[
        "Page Number"
    ]

    page_text = page_record[
        "Text"
    ]

    numeric_tokens = re.findall(
        r"\(?-?\d+(?:,\d{3})*(?:\.\d+)?\)?",
        page_text
    )

    money_tokens = re.findall(
        r"\$[\d,]+(?:\.\d+)?",
        page_text
    )

    percentage_tokens = re.findall(
        (
            r"\d+(?:\.\d+)?\s*percent"
            r"|\d+(?:\.\d+)?%"
        ),
        page_text,
        flags=re.IGNORECASE
    )

    page_characterisation.append({
        "Page Number":
            page_number,

        "Text Characters":
            len(page_text),

        "Word Count":
            len(
                page_text.split()
            ),

        "Numeric Token Count":
            len(
                numeric_tokens
            ),

        "Money Token Count":
            len(
                money_tokens
            ),

        "Percentage Token Count":
            len(
                percentage_tokens
            ),

        "Contains Table":
            "Table " in page_text,

        "Contains Chart":
            "Chart " in page_text,

        "Contains Technical Note":
            "Technical Note" in page_text,

        "Inside Reference Scope":
            (
                SOURCE_PAGE_START
                <= page_number
                <= SOURCE_PAGE_END
            )
    })


page_characterisation_df = (
    pd.DataFrame(
        page_characterisation
    )
)

display(
    page_characterisation_df
)

In [ ]:
# ============================================================
# 6. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract every explicitly stated occupational statistic from the headline
narrative sections on pages 1–5 of the BLS Occupational Employment and
Wages — May 2024 report.

For each statistic, return:

- Section
- Indicator
- Occupation or Group
- Value
- Unit
- Reference Period

Scope rules:

- Use only the headline narrative sections on pages 1–5.
- Do not extract the release identifier, release date, contact details,
  website addresses or other publication metadata.
- Do not extract general programme-description counts.
- Do not extract the Technical Note.
- Do not extract the full multi-page Table 1.
- Do not create separate observations from charts when the same values
  are already stated in the narrative.
- Extract only values explicitly stated in the source.
- Do not calculate, infer, reconstruct or increase the precision of a
  rounded value.
- Preserve rounded values in the reported scale. For example,
  “8.7 million” must be returned as Value 8.7 and Unit
  “million workers”.
- Return exact worker counts as numerical values with Unit “workers”.
- Return employment shares and concentration values with Unit “percent”.
- Return annual mean wages as numerical values with Unit “USD”.
- Preserve the occupation or occupational-group wording used in the
  headline narrative.
- Use “May 2024” as the Reference Period for every observation.
- Return the result as valid JSON using the exact field names defined
  in the extraction schema.
- Return one record for each selected statistic.
- Do not include explanations before or after the JSON.
"""

print(
    EXTRACTION_TASK
)

In [ ]:
# ============================================================
# 7. Reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "headline occupational statistic",

    "source_scope":
        "Headline narrative sections on pages 1–5",

    "fields": {
        "Section":
            (
                "Headline narrative section in which "
                "the statistic is reported."
            ),

        "Indicator":
            (
                "The type of explicitly reported "
                "employment, wage, share or "
                "concentration statistic."
            ),

        "Occupation or Group":
            (
                "The occupation, occupational group, "
                "industry, location or public-sector "
                "category associated with the value."
            ),

        "Value":
            (
                "The explicitly reported numerical "
                "value. Rounded values remain in their "
                "reported scale."
            ),

        "Unit":
            (
                "workers, million workers, percent "
                "or USD."
            ),

        "Reference Period":
            "May 2024.",

        "Source Location":
            (
                "The original PDF page supporting "
                "the observation."
            )
    },

    "excluded_content": [
        "Release identifier",
        "Release date",
        "Contact information",
        "General OEWS programme counts",
        "Technical Note",
        "Full Table 1",
        "Repeated chart-only copies",
        "Calculated or inferred values"
    ]
}


print(
    json.dumps(
        REFERENCE_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 8. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level": "headline occupational statistic",
    "expected_record_count": EXPECTED_REFERENCE_RECORD_COUNT,
    "fields": {
        "Section": {
            "type": ["string", "null"]
        },
        "Indicator": {
            "type": ["string", "null"]
        },
        "Occupation or Group": {
            "type": ["string", "null"]
        },
        "Value": {
            "type": ["number", "null"]
        },
        "Unit": {
            "type": ["string", "null"],
            "allowed_values": [
                "workers",
                "million workers",
                "percent",
                "USD"
            ]
        },
        "Reference Period": {
            "type": ["string", "null"],
            "expected_value": REFERENCE_PERIOD
        }
    },
    "expected_output_structure": {
        "document_id": DOCUMENT_ID,
        "records": [
            {
                "Section": "string or null",
                "Indicator": "string or null",
                "Occupation or Group": "string or null",
                "Value": "number or null",
                "Unit": (
                    "workers, million workers, "
                    "percent, USD or null"
                ),
                "Reference Period": "string or null"
            }
        ]
    }
}

print("\nExtraction schema:")
print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 9. Reference dataset
# ============================================================

reference_values = [
    # --------------------------------------------------------
    # Production occupations — page 1
    # --------------------------------------------------------
    {
        "Section": "Production occupations",
        "Indicator": "Employment",
        "Occupation or Group": "Production occupations",
        "Value": 8.7,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Share of national employment",
        "Occupation or Group": "Production occupations",
        "Value": 5.7,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Employment",
        "Occupation or Group": "miscellaneous assemblers and fabricators",
        "Value": 1.5,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Employment",
        "Occupation or Group": (
            "first-line supervisors of production "
            "and operating workers"
        ),
        "Value": 685140,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Employment",
        "Occupation or Group": (
            "inspectors, testers, sorters, samplers, "
            "and weighers"
        ),
        "Value": 591180,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Employment",
        "Occupation or Group": (
            "welders, cutters, solderers, and brazers"
        ),
        "Value": 424040,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "Production occupations",
        "Value": 50090,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "U.S. average wage",
        "Value": 67920,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "nuclear power reactor operators"
        ),
        "Value": 122830,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "power distributors and dispatchers"
        ),
        "Value": 109620,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "power plant operators",
        "Value": 95990,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 1"
    },

    # --------------------------------------------------------
    # Production occupations — page 2
    # --------------------------------------------------------
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "pressers, textile, garment, and related "
            "materials"
        ),
        "Value": 33370,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "laundry and dry-cleaning workers"
        ),
        "Value": 33990,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "hand sewers",
        "Value": 34810,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Share of state employment",
        "Occupation or Group": "Indiana",
        "Value": 11.3,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Share of state employment",
        "Occupation or Group": "Wisconsin",
        "Value": 10.5,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Share of area employment",
        "Occupation or Group": "Elkhart-Goshen, IN",
        "Value": 32.8,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Share of area employment",
        "Occupation or Group": "Sheboygan, WI",
        "Value": 22.3,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Share of area employment",
        "Occupation or Group": "Dalton, GA",
        "Value": 21.8,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "production occupations in petroleum "
            "and coal products manufacturing"
        ),
        "Value": 86760,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "production occupations in pulp, paper, "
            "and paperboard mills"
        ),
        "Value": 66230,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "production occupations in aerospace "
            "product and parts manufacturing"
        ),
        "Value": 66010,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },
    {
        "Section": "Production occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "production occupations in seafood "
            "product preparation and packaging"
        ),
        "Value": 38140,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 2"
    },

    # --------------------------------------------------------
    # Architecture and engineering occupations — page 3
    # --------------------------------------------------------
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Employment",
        "Occupation or Group": (
            "Architecture and engineering occupations"
        ),
        "Value": 2.6,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "Architecture and engineering occupations"
        ),
        "Value": 103980,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Employment",
        "Occupation or Group": "civil engineers",
        "Value": 355410,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Employment",
        "Occupation or Group": "industrial engineers",
        "Value": 350230,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Employment",
        "Occupation or Group": "mechanical engineers",
        "Value": 286760,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "computer hardware engineers",
        "Value": 156770,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "petroleum engineers",
        "Value": 153560,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "aerospace engineers",
        "Value": 141180,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "surveying and mapping technicians"
        ),
        "Value": 56890,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "environmental engineering technologists "
            "and technicians"
        ),
        "Value": 63070,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "drafters, all other",
        "Value": 66530,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Share of industry employment",
        "Occupation or Group": (
            "architecture and engineering occupations "
            "in architectural, engineering, and "
            "related services"
        ),
        "Value": 44.0,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Share of overall employment",
        "Occupation or Group": (
            "Architecture and engineering occupations"
        ),
        "Value": 1.7,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Share of industry employment",
        "Occupation or Group": (
            "architecture and engineering occupations "
            "in semiconductor and other electronic "
            "component manufacturing"
        ),
        "Value": 22.1,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Share of industry employment",
        "Occupation or Group": (
            "architecture and engineering occupations "
            "in aerospace product and parts manufacturing"
        ),
        "Value": 19.3,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 3"
    },

    # --------------------------------------------------------
    # Architecture and engineering occupations — page 4
    # --------------------------------------------------------
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Share of area employment",
        "Occupation or Group": "Lexington Park, MD",
        "Value": 8.1,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Share of area employment",
        "Occupation or Group": "Huntsville, AL",
        "Value": 7.7,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Architecture and engineering occupations",
        "Indicator": "Share of area employment",
        "Occupation or Group": "Columbus, IN",
        "Value": 7.1,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },

    # --------------------------------------------------------
    # Building and grounds cleaning and maintenance — page 4
    # --------------------------------------------------------
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Employment",
        "Occupation or Group": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Value": 4.5,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Value": 39540,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Employment",
        "Occupation or Group": (
            "janitors and cleaners, except maids "
            "and housekeeping cleaners"
        ),
        "Value": 2.2,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Employment",
        "Occupation or Group": (
            "landscaping and groundskeeping workers"
        ),
        "Value": 943430,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "first-line supervisors of landscaping, "
            "lawn service, and groundskeeping workers"
        ),
        "Value": 59380,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": "tree trimmers and pruners",
        "Value": 54970,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "maids and housekeeping cleaners"
        ),
        "Value": 36180,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "janitors and cleaners, except maids "
            "and housekeeping cleaners"
        ),
        "Value": 37460,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Employment",
        "Occupation or Group": (
            "building and grounds cleaning and "
            "maintenance jobs in services to buildings "
            "and dwellings"
        ),
        "Value": 1.8,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Employment",
        "Occupation or Group": "traveler accommodation",
        "Value": 517600,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Employment",
        "Occupation or Group": (
            "elementary and secondary schools"
        ),
        "Value": 354970,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "landscaping and groundskeeping workers nationally"
        ),
        "Value": 40880,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "landscaping and groundskeeping workers "
            "in West Virginia"
        ),
        "Value": 32310,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": (
            "Building and grounds cleaning and "
            "maintenance occupations"
        ),
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "landscaping and groundskeeping workers "
            "in Massachusetts and the District of Columbia"
        ),
        "Value": 49130,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },

    # --------------------------------------------------------
    # Largest occupations — page 4
    # --------------------------------------------------------
    {
        "Section": "Largest occupations",
        "Indicator": "Employment",
        "Occupation or Group": (
            "home health and personal care aides"
        ),
        "Value": 4.0,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Largest occupations",
        "Indicator": "Employment",
        "Occupation or Group": "retail salespersons",
        "Value": 3.8,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Largest occupations",
        "Indicator": "Employment",
        "Occupation or Group": (
            "fast food and counter workers"
        ),
        "Value": 3.8,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Largest occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "fast food and counter workers"
        ),
        "Value": 31350,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Largest occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "general office clerks",
        "Value": 45470,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Largest occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": "registered nurses",
        "Value": 98430,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Largest occupations",
        "Indicator": "Annual mean wage",
        "Occupation or Group": (
            "general and operations managers"
        ),
        "Value": 133120,
        "Unit": "USD",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },

    # --------------------------------------------------------
    # Public-sector occupations — pages 4–5
    # --------------------------------------------------------
    {
        "Section": "Public sector occupations",
        "Indicator": "Share of employment",
        "Occupation or Group": "public sector",
        "Value": 14.5,
        "Unit": "percent",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 4"
    },
    {
        "Section": "Public sector occupations",
        "Indicator": "Public-sector employment",
        "Occupation or Group": (
            "elementary school teachers, "
            "except special education"
        ),
        "Value": 1.2,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 5"
    },
    {
        "Section": "Public sector occupations",
        "Indicator": "Public-sector employment",
        "Occupation or Group": (
            "teaching assistants, except postsecondary"
        ),
        "Value": 1.1,
        "Unit": "million workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 5"
    },
    {
        "Section": "Public sector occupations",
        "Indicator": "Public-sector employment",
        "Occupation or Group": (
            "secondary school teachers, except special "
            "and career/technical education"
        ),
        "Value": 920980,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 5"
    },
    {
        "Section": "Public sector occupations",
        "Indicator": "Public-sector employment",
        "Occupation or Group": (
            "middle school teachers, except special "
            "and career/technical education"
        ),
        "Value": 546970,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 5"
    },
    {
        "Section": "Public sector occupations",
        "Indicator": "Public-sector employment",
        "Occupation or Group": (
            "police and sheriff's patrol officers"
        ),
        "Value": 660460,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 5"
    },
    {
        "Section": "Public sector occupations",
        "Indicator": "Public-sector employment",
        "Occupation or Group": "registered nurses",
        "Value": 533460,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 5"
    },
    {
        "Section": "Public sector occupations",
        "Indicator": "Public-sector employment",
        "Occupation or Group": (
            "janitors and cleaners, except maids "
            "and housekeeping cleaners"
        ),
        "Value": 493840,
        "Unit": "workers",
        "Reference Period": REFERENCE_PERIOD,
        "Source Location": "Page 5"
    }
]


reference_values_df = pd.DataFrame(
    reference_values,
    columns=REFERENCE_FIELDS
)

print(
    "Reference records:",
    len(reference_values_df)
)

display(
    reference_values_df
)

In [ ]:
# ============================================================
# 10. Validation of reference schema and record count
# ============================================================

actual_fields = (
    reference_values_df.columns.tolist()
)

missing_fields = [
    field
    for field in REFERENCE_FIELDS
    if field not in actual_fields
]

extra_fields = [
    field
    for field in actual_fields
    if field not in REFERENCE_FIELDS
]

reference_schema_valid = (
    not missing_fields
    and not extra_fields
)

reference_record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)


print(
    "Reference schema valid:",
    reference_schema_valid
)

print(
    "Missing fields:",
    missing_fields
)

print(
    "Extra fields:",
    extra_fields
)

print(
    "Reference record count valid:",
    reference_record_count_valid
)


if not reference_schema_valid:

    raise ValueError(
        "The reference schema is invalid."
    )


if not reference_record_count_valid:

    raise ValueError(
        "Unexpected reference record count: "
        f"{len(reference_values_df)}. "
        f"Expected "
        f"{EXPECTED_REFERENCE_RECORD_COUNT}."
    )

In [ ]:
# ============================================================
# 11. Validate values, units and field types
# ============================================================

missing_values_by_field = (
    reference_values_df[
        REFERENCE_FIELDS
    ]
    .isna()
    .sum()
    .to_dict()
)


unexpected_units = sorted(
    set(
        reference_values_df[
            "Unit"
        ].dropna()
    )
    - ALLOWED_UNITS
)


non_numeric_value_rows = (
    reference_values_df[
        ~reference_values_df[
            "Value"
        ].apply(
            lambda value:
                isinstance(
                    value,
                    (int, float)
                )
                and not isinstance(
                    value,
                    bool
                )
        )
    ]
)


incorrect_period_rows = (
    reference_values_df[
        reference_values_df[
            "Reference Period"
        ]
        != REFERENCE_PERIOD
    ]
)


print(
    "Missing values by field:"
)

print(
    json.dumps(
        missing_values_by_field,
        indent=2
    )
)

print(
    "Unexpected units:",
    unexpected_units
)

print(
    "Non-numeric value rows:",
    len(non_numeric_value_rows)
)

print(
    "Incorrect reference-period rows:",
    len(incorrect_period_rows)
)


if any(
    count > 0
    for count
    in missing_values_by_field.values()
):

    raise ValueError(
        "The reference values contain missing "
        "mandatory fields."
    )


if unexpected_units:

    raise ValueError(
        f"Unexpected units: {unexpected_units}"
    )


if not non_numeric_value_rows.empty:

    raise ValueError(
        "All reference values must be numerical."
    )


if not incorrect_period_rows.empty:

    raise ValueError(
        "All records must use May 2024 as the "
        "reference period."
    )

In [ ]:
# ============================================================
# 12. Validation of source locations
# ============================================================

def extract_page_number(
    source_location
):
    """
    Extract the page number from a value such as 'Page 3'.
    """

    match = re.fullmatch(
        r"Page\s+(\d+)",
        str(
            source_location
        ).strip()
    )

    if not match:

        return None

    return int(
        match.group(1)
    )


reference_values_df[
    "_Source Page"
] = reference_values_df[
    "Source Location"
].apply(
    extract_page_number
)


invalid_source_locations = (
    reference_values_df[
        reference_values_df[
            "_Source Page"
        ].isna()
    ]
)


out_of_scope_locations = (
    reference_values_df[
        ~reference_values_df[
            "_Source Page"
        ].between(
            SOURCE_PAGE_START,
            SOURCE_PAGE_END
        )
    ]
)


print(
    "Invalid source locations:",
    len(invalid_source_locations)
)

print(
    "Out-of-scope source locations:",
    len(out_of_scope_locations)
)


if not invalid_source_locations.empty:

    raise ValueError(
        "One or more source locations do not "
        "follow the 'Page N' format."
    )


if not out_of_scope_locations.empty:

    raise ValueError(
        "One or more reference records fall "
        "outside pages 1–5."
    )


reference_values_df = (
    reference_values_df.drop(
        columns=[
            "_Source Page"
        ]
    )
)

In [ ]:
# ============================================================
# 13. Checks for duplicate reference keys
# ============================================================

REFERENCE_KEY_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Reference Period"
]


duplicate_reference_mask = (
    reference_values_df.duplicated(
        subset=REFERENCE_KEY_FIELDS,
        keep=False
    )
)


duplicate_reference_records = (
    reference_values_df[
        duplicate_reference_mask
    ]
    .sort_values(
        REFERENCE_KEY_FIELDS
    )
)


print(
    "Duplicate reference records:",
    len(
        duplicate_reference_records
    )
)


if not duplicate_reference_records.empty:

    display(
        duplicate_reference_records
    )

    raise ValueError(
        "Duplicate reference keys were detected."
    )

In [ ]:
# ============================================================
# 14. Validation of reference scope exclusions
# ============================================================

excluded_indicators = {
    "Release reference",
    "Release date",
    "Contact information"
}


observed_excluded_indicators = sorted(
    set(
        reference_values_df[
            "Indicator"
        ]
    )
    & excluded_indicators
)


print(
    "Excluded indicators found:",
    observed_excluded_indicators
)


if observed_excluded_indicators:

    raise ValueError(
        "Release metadata must not be included "
        "in the reference-value dataset."
    )

In [ ]:
# ============================================================
# 15. Reference summary
# ============================================================

REFERENCE_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "reference_scope":
        "Headline narrative sections on pages 1–5",

    "number_of_reference_records":
        int(
            len(reference_values_df)
        ),

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "schema_valid":
        bool(
            reference_schema_valid
        ),

    "sections":
        sorted(
            reference_values_df[
                "Section"
            ].unique().tolist()
        ),

    "indicators":
        sorted(
            reference_values_df[
                "Indicator"
            ].unique().tolist()
        ),

    "units":
        sorted(
            reference_values_df[
                "Unit"
            ].unique().tolist()
        ),

    "records_by_page":
        (
            reference_values_df[
                "Source Location"
            ]
            .value_counts()
            .sort_index()
            .to_dict()
        ),

    "records_by_section":
        (
            reference_values_df[
                "Section"
            ]
            .value_counts()
            .sort_index()
            .to_dict()
        ),

    "missing_values_by_field":
        missing_values_by_field,

    "duplicate_reference_key_count":
        int(
            len(
                duplicate_reference_records
            )
        ),

    "reference_values_manually_constructed":
        True,

    "reference_values_branch_independent":
        True,

    "reference_values_reused_across_branches":
        True,

    "rounded_values_preserved_in_reported_scale":
        True
}


print(
    json.dumps(
        REFERENCE_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 16. Document characterisation
# ============================================================

numeric_tokens = re.findall(
    r"\(?-?\d+(?:,\d{3})*(?:\.\d+)?\)?",
    full_text
)

word_count = len(
    full_text.split()
)

numeric_token_to_word_ratio = (
    len(numeric_tokens) / word_count
    if word_count
    else 0
)

money_tokens = re.findall(
    r"\$[\d,]+(?:\.\d+)?",
    full_text
)

percentage_tokens = re.findall(
    (
        r"\d+(?:\.\d+)?\s*percent"
        r"|\d+(?:\.\d+)?%"
    ),
    full_text,
    flags=re.IGNORECASE
)


DOCUMENT_CHARACTERISATION = {
    "document_id":
        DOCUMENT_ID,

    "page_count":
        len(doc),

    "text_extractable":
        True,

    "ocr_required":
        False,

    "line_count":
        len([
            line
            for line
            in full_text.splitlines()
            if line.strip()
        ]),

    "numeric_token_count":
        len(
            numeric_tokens
        ),

    "numeric_token_to_word_ratio":
        round(
            numeric_token_to_word_ratio,
            3
        ),

    "money_token_count":
        len(
            money_tokens
        ),

    "percentage_token_count":
        len(
            percentage_tokens
        ),

    "contains_headline_narrative":
        True,

    "contains_charts":
        (
            "Chart 1" in full_text
            and "Chart 2" in full_text
        ),

    "contains_multi_page_statistical_table":
        (
            "Table 1." in full_text
        ),

    "contains_technical_note":
        (
            "Technical Note" in full_text
        ),

    "contains_footnotes":
        (
            "footnotes" in full_text.lower()
        ),

    "reference_scope_pages":
        [
            SOURCE_PAGE_START,
            SOURCE_PAGE_END
        ],

    "table_1_scope_excluded":
        True,

    "detected_years":
        sorted(
            set(
                re.findall(
                    r"\b20\d{2}\b",
                    full_text
                )
            )
        )
}


print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 17. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "Medium",
        "Evidence Source":
            "PDF text extraction + manual inspection",
        "Justification":
            "Narrative sections generally follow a coherent reading "
            "sequence, but the document combines prose, charts, "
            "technical sections, and multi-page statistical tables "
            "that complicate the overall reading order."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "High",
        "Evidence Source":
            "PDF table inspection + extracted text inspection",
        "Justification":
            "Table 1 spans pages 8–23 and contains repeated headers, "
            "hierarchical occupation labels, continuation structures, "
            "footnotes, and dense multi-column numerical data whose "
            "relationships are not always preserved in linear text."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Medium",
        "Evidence Source":
            "Manual document inspection",
        "Justification":
            "Major sections are identifiable, but the document "
            "contains several hierarchy levels across narrative "
            "sections, chart titles, technical-note subsections, "
            "occupational groups, and continued table headers."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "The PDF is visually clear and text and numerical content "
            "are readable throughout the document."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "No relevant scanning noise, blur, or degradation affects "
            "document readability."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "Low",
        "Evidence Source":
            "Automated PDF text extraction",
        "Justification":
            "Relevant content is embedded as machine-readable text "
            "and OCR is not required."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Low",
        "Evidence Source":
            "Manual content inspection",
        "Justification":
            "Occupational, employment, wage, industry, and geographic "
            "terminology is used consistently throughout the report, "
            "with clearly established statistical naming conventions."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "Medium",
        "Evidence Source":
            "Reference-schema comparison",
        "Justification":
            "The extraction schema captures the selected headline "
            "statistics, but the generic 'Occupation or Group' field "
            "must represent multiple semantic entity types."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "High",
        "Evidence Source":
            "Automated text profiling + manual inspection",
        "Justification":
            "The document contains a very high concentration of "
            "employment counts, wages, percentages, and statistical "
            "table values across the 23 pages."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source":
            "Reference-value verification",
        "Justification":
            "All information required for the defined pages 1–5 "
            "extraction scope is present in the source and represented "
            "in the 70-record reference dataset."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Medium",
        "Evidence Source":
            "Manual source and reference inspection",
        "Justification":
            "The document is internally coherent, but some statistics "
            "appear at different precision levels across narrative, "
            "charts, and Table 1, creating comparison complexity "
            "without representing true contradictions."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "High",
        "Evidence Source":
            "Document profiling + manual inspection",
        "Justification":
            "The report combines narrative prose, bullet lists, "
            "charts, technical notes, a dense multi-page statistical "
            "table, repeated headers, and footnotes."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "Medium",
        "Evidence Source":
            "Reference and source inspection",
        "Justification":
            "The extraction scope contains several units including "
            "workers, million workers, percent, and USD, together with "
            "long occupational, industry, and geographic labels."
    }
]

indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

indicator_assessment_df

In [ ]:
# ============================================================
# 18. Validation of indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores detected: "
        f"{invalid_scores}"
    )


expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if missing_indicators:
    raise ValueError(
        f"Missing required indicators: "
        f"{missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators detected: "
        f"{unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

In [ ]:
# ============================================================
# 19. Dimension-level quality assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

dimension_assessment_df

In [ ]:
# ============================================================
# 20. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}


print(
    json.dumps(
        QUALITY_EVIDENCE,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 21. Reference-construction metadata
# ============================================================

REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "reference_file":
        "D3_reference_values.csv",

    "reference_construction_method":
        "Manual document-grounded construction",

    "reference_scope":
        "Headline narrative sections on pages 1–5",

    "reference_period":
        REFERENCE_PERIOD,

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        REFERENCE_FIELDS,

    "allowed_units":
        sorted(
            ALLOWED_UNITS
        ),

    "rounded_values_preserved":
        True,

    "rounded_values_expanded_to_exact_counts":
        False,

    "table_1_used_to_replace_narrative_values":
        False,

    "release_metadata_included_as_records":
        False,

    "manual_calculation_applied":
        False,

    "semantic_inference_applied":
        False,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "notes": (
        "The fixed D3 reference dataset contains every "
        "explicitly stated occupational statistic selected "
        "under the controlled pages 1–5 narrative scope. "
        "Release metadata, methodology content, charts that "
        "repeat narrative values and the full Table 1 are "
        "excluded. Rounded narrative values remain in their "
        "reported scale."
    )
}


print(
    json.dumps(
        REFERENCE_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 22. Export Stage 1 outputs
# ============================================================

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR
    / "D3_reference_values.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR
    / "D3_reference_values.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR
    / "D3_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR
    / "D3_dimension_assessment.csv"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR
    / "D3_extraction_schema.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR
    / "D3_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D3_reference_summary.json"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR
    / "D3_reference_metadata.json"
)

DOCUMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D3_document_metadata.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR
    / "D3_document_characterisation.json"
)

PAGE_CHARACTERISATION_PATH = (
    OUTPUT_DIR
    / "D3_page_characterisation.csv"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR
    / "D3_quality_evidence.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR
    / "D3_extraction_task.txt"
)


reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False
)

reference_values_df.to_json(
    REFERENCE_VALUES_JSON_PATH,
    orient="records",
    indent=2,
    force_ascii=False
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False
)

page_characterisation_df.to_csv(
    PAGE_CHARACTERISATION_PATH,
    index=False
)


json_outputs = [
    (
        EXTRACTION_SCHEMA_PATH,
        EXTRACTION_SCHEMA
    ),
    (
        REFERENCE_SCHEMA_PATH,
        REFERENCE_SCHEMA
    ),
    (
        REFERENCE_SUMMARY_PATH,
        REFERENCE_SUMMARY
    ),
    (
        REFERENCE_METADATA_PATH,
        REFERENCE_METADATA
    ),
    (
        DOCUMENT_METADATA_PATH,
        DOCUMENT_METADATA
    ),
    (
        DOCUMENT_CHARACTERISATION_PATH,
        DOCUMENT_CHARACTERISATION
    ),
    (
        QUALITY_EVIDENCE_PATH,
        QUALITY_EVIDENCE
    )
]


for output_path, content in json_outputs:

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            content,
            file,
            indent=2,
            ensure_ascii=False
        )


with open(
    EXTRACTION_TASK_PATH,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        EXTRACTION_TASK.strip()
    )


print(
    "D3 Stage 1 outputs exported."
)

In [ ]:
# ============================================================
# 23. Final integrity conclusion
# ============================================================

REFERENCE_INTEGRITY_PASSED = all([
    reference_schema_valid,
    reference_record_count_valid,
    not unexpected_units,
    non_numeric_value_rows.empty,
    incorrect_period_rows.empty,
    invalid_source_locations.empty,
    out_of_scope_locations.empty,
    duplicate_reference_records.empty,
    not observed_excluded_indicators,
    all(
        count == 0
        for count
        in missing_values_by_field.values()
    )
])


REFERENCE_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "reference_integrity_passed":
        bool(
            REFERENCE_INTEGRITY_PASSED
        ),

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_records":
        int(
            len(reference_values_df)
        ),

    "schema_valid":
        bool(
            reference_schema_valid
        ),

    "record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "missing_mandatory_values":
        int(
            sum(
                missing_values_by_field.values()
            )
        ),

    "unexpected_unit_count":
        int(
            len(unexpected_units)
        ),

    "duplicate_reference_key_count":
        int(
            len(
                duplicate_reference_records
            )
        ),

    "invalid_source_location_count":
        int(
            len(
                invalid_source_locations
            )
        ),

    "out_of_scope_source_location_count":
        int(
            len(
                out_of_scope_locations
            )
        ),

    "excluded_metadata_record_count":
        int(
            len(
                observed_excluded_indicators
            )
        )
}


REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D3_reference_integrity.json"
)


with open(
    REFERENCE_INTEGRITY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        REFERENCE_INTEGRITY,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        REFERENCE_INTEGRITY,
        indent=2,
        ensure_ascii=False
    )
)


if not REFERENCE_INTEGRITY_PASSED:

    raise ValueError(
        "The D3 reference dataset failed "
        "the integrity checks."
    )

In [ ]:
# ============================================================
# 24. List generated outputs
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    EXTRACTION_SCHEMA_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    REFERENCE_INTEGRITY_PATH,
    DOCUMENT_METADATA_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    PAGE_CHARACTERISATION_PATH,
    QUALITY_EVIDENCE_PATH,
    EXTRACTION_TASK_PATH
]


print(
    "Generated D3 Stage 1 files:\n"
)

for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name
    )